In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


final script generated

In [ ]:

# Basic paths
proj_path = "/mnt/1e99f03a-239b-4885-af31-60eb1322a5e1/IEL/sklearn"
roi_list = ["LeftAmyg"] #, "RightAmyg"

# Subject ID
def participant_id(subj):
    proj = "IEL"
    subj = f"{subj:03d}"
    return f"{proj}{subj}"


# Subect specific paths
def input_path(subj, roi_name):
    part_id = participant_id(subj)
    return os.path.join(proj_path, "data", "individual", part_id, "BetaSeries", "ROI",
                        f"{part_id}_{roi_name}_betas.csv")

def output_path(subj):
    part_id = participant_id(subj)
    return os.path.join(proj_path, "data", "individual", part_id,"MVPA","ROI")

# Decoding class pairs 
class_pairs = [("PositiveLoss", "PositiveNoLoss"), ("NeutralLoss", "NeutralNoLoss")]

# Some parameters and model settings
hyperparameter_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
cv = LeaveOneGroupOut()
scaler = StandardScaler()
n_permutations = 1000
all_subject_summaries = []

for subj in range(1,2):

    participant = participant_id(subj)

    for roi_name in roi_list:

        print(participant, roi_name)

        roi_file = input_path(subj, roi_name)
        df = pd.read_csv(roi_file, sep="\t")

        df["Condition"] = df["Emotion"].astype(str) + df["Loss"].astype(str)

        voxel_matrix = []
        for col in df.columns:
            if col.lower().startswith('voxel'):
                voxel_matrix.append(col)

        # Class pair loop
        for class1, class2 in class_pairs:

            pair_name = f"{class1}_vs_{class2}"
            print("\nRunning:", pair_name)

            filtered_df = df[df["Condition"].isin([class1, class2])].copy()

            voxel = filtered_df[voxel_matrix].values
            label = filtered_df["Condition"].values
            groups = filtered_df["Trial"].values

            fold_results = []
            cv_qc = []
            fold_num = 1

            all_true = np.array([])
            all_pred = np.array([])
            for train_idx, test_idx in cv.split(voxel, label, groups):

                train_voxelX = voxel[train_idx]
                test_voxelX = voxel[test_idx]
                train_labelY = label[train_idx]
                test_conditionY = label[test_idx]

                best_c = None
                best_score = -np.inf

                # Hyperparameter tuning using GridsearchCV - non vectorised syntax
                for current_c in hyperparameter_grid:

                    tuning_scores = []
                    tuning_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

                    for c_i, c_j in tuning_CV.split(train_voxelX, train_labelY):

                        x_train = train_voxelX[c_i]
                        y_train = train_labelY[c_i]
                        x_test = train_voxelX[c_j]
                        y_test = train_labelY[c_j]

                        scaler_tune = StandardScaler()

                        x_train = scaler_tune.fit_transform(x_train)
                        x_test = scaler_tune.transform(x_test)

                        model = LinearSVC(C=current_c, max_iter=10000)
                        model.fit(x_train, y_train)
                        prediction = model.predict(x_test)

                        tuning_scores.append(accuracy_score(y_test, prediction))

                    mean_score = np.mean(tuning_scores)

                    if mean_score > best_score:
                        best_score = mean_score
                        best_c = current_c

                # QC table
                fold_summary = filtered_df.copy()
                fold_summary["Fold"] = fold_num
                fold_summary["Type"] = "Train"
                fold_summary.loc[filtered_df.index[test_idx],"Type"] = "Test"
                cv_qc.append(fold_summary )
                
                scaler_fold = StandardScaler()
                train_voxelX = scaler_fold.fit_transform(train_voxelX)
                test_voxelX = scaler_fold.transform(test_voxelX)

                final_model = LinearSVC(C=best_c, max_iter=10000)
                final_model.fit(train_voxelX, train_labelY)
                prediction = final_model.predict(test_voxelX)

                accuracy = accuracy_score(test_conditionY, prediction)

                fold_results.append({"Fold": fold_num,
                    "Accuracy": accuracy,
                    "Best_C": best_c})

                fold_num += 1
                    
                all_true = np.concatenate([all_true, test_conditionY])
                all_pred = np.concatenate([all_pred, prediction])

            fold_results = pd.DataFrame(fold_results)
            cv_qc = pd.concat(cv_qc, ignore_index=True)
            
            # Confusion matrix
            cm = confusion_matrix(
                all_true,
                all_pred,
                labels=[class1, class2])

            # Ooutput path
            output_dir = output_path(subj)
            pair_output = os.path.join(output_dir, roi_name, pair_name)
            os.makedirs(pair_output, exist_ok=True)

            # Permutation testing
            real_accuracy = fold_results["Accuracy"].mean()

            permutation_accuracies = []

            for perm in range(n_permutations):

                shuffled_label = np.random.permutation(label)
                perm_fold_accuracy = []

                for train_idx, test_idx in cv.split(voxel, shuffled_label, groups):

                    train_voxelX = voxel[train_idx]
                    test_voxelX = voxel[test_idx]

                    train_labelY = shuffled_label[train_idx]
                    test_labelY = shuffled_label[test_idx]

                    scaler_perm = StandardScaler()
                    train_voxelX = scaler_perm.fit_transform(train_voxelX)
                    test_voxelX = scaler_perm.transform(test_voxelX)

                    perm_model = LinearSVC(C=best_c, max_iter=10000)
                    perm_model.fit(train_voxelX, train_labelY)

                    perm_prediction = perm_model.predict(test_voxelX)

                    perm_fold_accuracy.append(
                        accuracy_score(test_labelY, perm_prediction))

                permutation_accuracies.append(np.mean(perm_fold_accuracy))

            permutation_distribution = pd.DataFrame({
                "PermutationAccuracy": permutation_accuracies})

            p_value = (np.sum(permutation_accuracies >= real_accuracy) + 1) / (n_permutations + 1)

            summary = pd.DataFrame([{"Participant": participant,
                "ROI": roi_name,
                "ClassPair": pair_name,
                "MeanAccuracy": real_accuracy,
                "P_value": p_value}])

            # Save (TSV)
            fold_results.to_csv(os.path.join(pair_output, "fold_results.tsv"), sep="\t", index=False)
            cv_qc.to_csv(os.path.join(pair_output, "CV_QC.tsv"), sep="\t", index=False)
            permutation_distribution.to_csv(os.path.join(pair_output, "permutation_distribution.tsv"), sep="\t", index=False)
            summary.to_csv(os.path.join(pair_output, "summary.tsv"), sep="\t", index=False)

            all_subject_summaries.append(summary)

            # Visualisations
            # Permutation distribution with real decoder accuracy
            plt.hist(permutation_accuracies, bins=30)
            plt.axvline(real_accuracy,
                color="red",
                linewidth=2,
                label=f"Observed Accuracy = {real_accuracy:.3f}")

            plt.xlabel("Accuracy")
            plt.ylabel("Count")
            plt.legend()
            plt.show()
            plt.savefig(
            os.path.join(pair_output, "permutation_distribution.png"),dpi=300,bbox_inches="tight")

            # confusion matrix
            plt.figure(figsize=(4,4))
            plt.imshow(cm)
            plt.xticks([0,1], [class1, class2], rotation=45)
            plt.yticks([0,1], [class1, class2])
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title(pair_name)

            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(j, i, cm[i,j], ha="center", va="center")

            plt.tight_layout()
            plt.savefig(os.path.join(pair_output, "confusion_matrix.png"), dpi=300,bbox_inches="tight")
            plt.close()


# Group accuracy
all_subject_summaries = pd.concat(all_subject_summaries, ignore_index=True)
print(all_subject_summaries["MeanAccuracy"].mean())